# 3D penalty contact with `ContactGap`

This example solves a frictionless 3D contact problem with the contact weak form

$$
\delta W_c
=
\int_{\Gamma_c}
\delta d_n\,c_n\,d_n\,\mathrm d\Gamma .
$$

The contact kinematic operator is evaluated directly at the slave-side Gauss
points. For a frozen contact geometry,

$$
d_n = G_q x,\qquad x=r+u,
$$

therefore

$$
K_c
=
\int_{\Gamma_c}
G_q^T c_n G_q\,\mathrm d\Gamma,
\qquad
r_c = K_c(r+u).
$$

In LowLevelFEM notation this becomes simply

```julia
G  = ContactGap(C)
Kc = ∫(G ⋅ cn ⋅ G)
```

The closest-point projection, contact normal, active Gauss points and the
contact tangent are refreshed during the nonlinear iteration.


In [1]:
using LowLevelFEM, LinearAlgebra, SparseArrays


[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (caches not reused: 2 for different dependency version already loaded, 7 for file size changed)
Precompiling packages...
   0.0 s  ✓ LibCURL_jll
  10.0 s  ✓ LowLevelFEM
  1 dependency successfully precompiled in 11 seconds. 134 already precompiled.


## Geometry and finite-element field


In [2]:
openGeometry("boxes.geo")
# openPreProcessor()


In [3]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u)


Problem("boxes", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 26278, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false)

## Initial displacement without contact

The prescribed displacement is first applied without contact. This gives a
convenient starting configuration for the nonlinear contact iteration.


In [4]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition(
    "top",
    ux=0,
    uz=0,
    uy=(x,y,z)->-x*(x-10) * z*(z-10) / 4250
)

support = [bc_bottom, bc_top]

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

@time u0 = solveField(K, f, support=support)

showDoFResults(u0, name="u0", factor=1, visible=false)


 15.880537 seconds (16.06 M allocations: 1.970 GiB, 2.57% gc time, 71.55% compilation time)


0

## Contact definition

`Contact` stores the current slave-master geometry and search data.
`ContactGap(C)` is the weak-form contact operator.

For frictionless penalty contact only the normal component is used. The penalty
parameter `cn` has units of traction per gap.


In [5]:
cn = 1e7

C = contact(
    U,
    displacement=u0,
    master="master",
    slave="slave",
    topology_tol=0.01
)

G = ContactGap(C)

# Reference nodal position vector. The absolute current position is r + u.
r = nodePositionVector(U)


nodal VectorField
[0.0; -2.0; … ; 1.795491126453324; 0.4787301686243378;;]

## Nonlinear penalty iteration

At each iteration the contact geometry is updated, then the contact tangent and
residual are assembled from the same frozen configuration:

$$
R(u)=Ku-f+K_c(u)(r+u).
$$

The correction satisfies

$$
(K+K_c)\Delta u=-R.
$$

The increment has homogeneous essential boundary conditions. A backtracking
line search is used because the closest-point projection and the active contact
set may change after the correction.


In [6]:
support_increment = [
    BoundaryCondition("bottom", ux=0, uy=0, uz=0),
    BoundaryCondition("top",    ux=0, uy=0, uz=0)
]

free = freeDoFs(U, support_increment)

# Residual norm on unconstrained displacement DoFs only.
freeNorm(v::VectorField) =
    LinearAlgebra.norm(elementsToNodes(v).a[free, 1])

maxiter = 30

# For the penalty iteration these tolerances are sufficient for now.
tol_u = 1e-3
tol_R = 1e-3

# Fixed relaxation parameter.
ω = 0.5

u_it = copy(u0)

Rref = 0.0
converged = false

for iter in 1:maxiter

    # ------------------------------------------------------------
    # 1. Update current contact geometry
    # ------------------------------------------------------------
    updateContact!(C, u_it)

    # ------------------------------------------------------------
    # 2. Assemble contact tangent and contact residual
    # ------------------------------------------------------------
    Kc = ∫(G ⋅ cn ⋅ G, gauss=2)

    rc = Kc * (r + u_it)

    # Total equilibrium residual
    R = K * u_it - f + rc

    R0 = freeNorm(R)

    if iter == 1
        Rref = max(R0, eps(Float64))
    end

    # ------------------------------------------------------------
    # 3. Frozen-contact Newton correction
    # ------------------------------------------------------------
    Δu = solveField(
        K + Kc,
        -R,
        support=support_increment
    )

    # ------------------------------------------------------------
    # 4. Convergence measures
    # ------------------------------------------------------------
    err_R = R0 / Rref

    err_u =
        freeNorm(Δu) /
        max(freeNorm(u_it), eps(Float64))

    println(
        "iter = ", iter,
        ", active nodes = ", count(C.active),
        ", min gap = ", minimum(C.gap_values),
        ", |R|/|R0| = ", err_R,
        ", |du|/|u| = ", err_u
    )

    # ------------------------------------------------------------
    # 5. Convergence check
    # ------------------------------------------------------------
    if err_R < tol_R && err_u < tol_u
        converged = true
        break
    end

    # ------------------------------------------------------------
    # 6. Relaxed update
    # ------------------------------------------------------------
    u_it = u_it + ω * Δu
end

# Final solution
u = u_it

# Synchronize Contact with the final displacement.
updateContact!(C, u)

println("converged = ", converged)
println("final min gap = ", minimum(C.gap_values))

iter = 1, active nodes = 337, min gap = -0.031676157250614345, |R|/|R0| = 1.0, |du|/|u| = 0.047594419564204446
iter = 2, active nodes = 333, min gap = -0.01594536340791612, |R|/|R0| = 0.4996069129348161, |du|/|u| = 0.0240370590536943
iter = 3, active nodes = 325, min gap = -0.008080076385209977, |R|/|R0| = 0.24970456490827397, |du|/|u| = 0.012083141936513688
iter = 4, active nodes = 322, min gap = -0.004147495667664609, |R|/|R0| = 0.124837435295471, |du|/|u| = 0.0061876359129417104
iter = 5, active nodes = 316, min gap = -0.002181272025105767, |R|/|R0| = 0.062439682214082685, |du|/|u| = 0.003456262501786204
iter = 6, active nodes = 298, min gap = -0.0011982331834092237, |R|/|R0| = 0.03123607943785897, |du|/|u| = 0.0018616428926682567
iter = 7, active nodes = 290, min gap = -0.0007067393489830515, |R|/|R0| = 0.01564256623720162, |du|/|u| = 0.0011971724510571554
iter = 8, active nodes = 280, min gap = -0.0004610219332679496, |R|/|R0| = 0.007827905719654457, |du|/|u| = 0.00074259840838405

## Final gap and displacement

`ContactGap(C, u)` evaluates the current normal gap as an ordinary
`ScalarField`. It can therefore be passed directly to the standard LowLevelFEM
postprocessing functions.


In [7]:
gap = ContactGap(C, u)

gap_slave = nodesToElements(
    gap,
    onPhysicalGroup="slave"
)

showDoFResults(
    u,
    name="u",
    factor=1,
    visible=false
)

showElementResults(
    gap_slave,
    name="gap",
    visible=true
)


2

## Penetration and penalty pressure

For the frictionless penalty law,

$$
p_n =
\begin{cases}
-c_n d_n, & d_n < 0,\\
0, & d_n \ge 0.
\end{cases}
$$

The following field is a nodal/element-interpolated visualization of the
penalty pressure. The actual weak form is assembled at Gauss points.


In [8]:
penetration = mapScalarField(
    g -> min(g, 0.0),
    gap_slave
)

pressure = mapScalarField(
    g -> max(-cn * g, 0.0),
    gap_slave
)

showElementResults(
    penetration,
    name="penetration",
    visible=false
)

showElementResults(
    pressure,
    name="contact pressure",
    visible=true
)


4

In [ ]:
updateContact!(C, u)

gap_nodal = ContactGap(C, u)
gap_gauss = ContactGap(C, u; gauss=2)

showElementResults(
    nodesToElements(gap_nodal, onPhysicalGroup="slave"),
    name="nodal gap",
    visible=false
)

showElementResults(
    gap_gauss,
    name="Gauss gap",
    visible=true
)

## Optional: inspect one frozen contact tangent

The matrix below is the contact tangent corresponding to the final frozen
contact geometry and active Gauss-point set.


In [ ]:
Kc = ∫(G ⋅ cn ⋅ G)

println("size(Kc) = ", size(Kc))
println("nnz(Kc)  = ", nnz(Kc.A))


size(Kc) = (78834, 78834)
nnz(Kc)  = 134766


In [ ]:
openPostProcessor()


## Extension to normal + tangential penalty

The same operator can return all local contact components. In 3D their ordering
is `(n,t1,t2)`:

```julia
Gall = ContactGap(C; components=:all)
Dc   = ContactStiffness(C, cn; ct=ct)

Kc = ∫(Gall ⋅ Dc ⋅ Gall)
```

For frictionless contact use the scalar normal formulation above. A nonzero
`ct` is a tangential penalty stiffness; a physical Coulomb friction model
requires an additional tangential history/stick-slip law.
